In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns

from imblearn.over_sampling import SMOTE
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix


In [70]:
pd.set_option('display.max_column',None)

In [ ]:
df = pd.read_csv("../data/processed/cleaned_data_v4.csv")

In [72]:
df.head()

,transaction_date,amount,ip_address_risk_score,device_trust_score,avg_amount_last_24h,device_change_flag,location_change_flag,merchant_historical_fraud_rate,cust_txn_count,cust_fraud_count,cust_fraud_rate,cust_avg_amt,device_count,merchant_fraud_rate,combined_risk,month,day,is_fraud
0,2024-01-01,556.63,0.041787,0.841375,3294.64,0,0,0.063596,11,0,0.0,9659.361818,11,0.073620,0.029911,1,1,0
1,2024-01-01,10158.89,0.162148,0.119578,4163.76,1,0,0.057505,11,0,0.0,9659.361818,11,0.099448,0.094997,1,1,0
2,2024-01-10,15754.57,0.774662,0.718447,5077.26,0,1,0.053729,11,0,0.0,9659.361818,11,0.077844,0.102428,1,10,0
3,2024-01-20,6095.68,0.259900,0.069586,1187.30,0,0,0.004334,11,0,0.0,9659.361818,11,0.086486,0.032338,1,20,0
4,2024-01-23,15324.24,0.376711,0.286630,10936.02,0,0,0.131218,11,0,0.0,9659.361818,11,0.096386,0.086691,1,23,0


In [73]:
df.shape

(50000, 18)

In [74]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 18 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   transaction_date                50000 non-null  object 
 1   amount                          50000 non-null  float64
 2   ip_address_risk_score           50000 non-null  float64
 3   device_trust_score              50000 non-null  float64
 4   avg_amount_last_24h             50000 non-null  float64
 5   device_change_flag              50000 non-null  int64  
 6   location_change_flag            50000 non-null  int64  
 7   merchant_historical_fraud_rate  50000 non-null  float64
 8   cust_txn_count                  50000 non-null  int64  
 9   cust_fraud_count                50000 non-null  int64  
 10  cust_fraud_rate                 50000 non-null  float64
 11  cust_avg_amt                    50000 non-null  float64
 12  device_count                    

In [75]:
df['transaction_date'] = pd.to_datetime(df['transaction_date'])

### Building pipeline

In [76]:
numerical_col = df.select_dtypes(['int','float']).columns.to_list()

if 'is_fraud' in numerical_col:
    numerical_col.remove('is_fraud')

In [77]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num',StandardScaler(),numerical_col)
    ],
    remainder='passthrough'
)

In [ ]:
def scorer(data, model_name, model):
    
    data = data.copy()
    
    # Convert to datetime
    data['transaction_date'] = pd.to_datetime(data['transaction_date'])
    
    
    # Dividing the train and test data based on transaction date to prevent Data Leakage
    train = data[data['transaction_date'] < '2024-03-01'].copy()
    test  = data[data['transaction_date'] >= '2024-03-01'].copy()
    
    train = train.drop(columns=['transaction_date'])
    test  = test.drop(columns=['transaction_date'])
    
    
    # Train test split
    X_train = train.drop('is_fraud', axis=1)
    y_train = train['is_fraud']

    X_test = test.drop('is_fraud', axis=1)
    y_test = test['is_fraud']
    
    
    # Doing sampling for balancing the data using SMOTE
    smote = SMOTE(random_state = 42)
    X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
    
    
    # initialize Pipeline
    pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('classifier',model)
    ])
    
    
    # Fitting the Pipeline
    pipeline.fit(X_train_sm,y_train_sm)
    
    # Predict using pipeline
    y_pred = pipeline.predict(X_test)

    if hasattr(pipeline, "predict_proba"):
        y_prob = pipeline.predict_proba(X_test)[:, 1]
    else:
        y_prob = pipeline.decision_function(X_test)
    
    
    # Calculating the metrics
    recall = recall_score(y_test, y_pred)

    f1 = f1_score(y_test, y_pred)

    rou_auc = roc_auc_score(y_test, y_prob)
    
    
    output = {
        'Models' : model_name,
        'Recall' : recall,
        'F1-score' : f1,
        'ROU-AUC' : rou_auc
    }
    
    return output

In [79]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

model_dict = {
    'svm' : SVC(probability=True),
    'decision_tree' : DecisionTreeClassifier(),
    'random_forest' : RandomForestClassifier(),
    'extra_trees' : ExtraTreesClassifier(),
    'gradient_boosting' : GradientBoostingClassifier(),
    'adaboost' : AdaBoostClassifier(),
    'mlp' : MLPClassifier(),
    'xgboost' : XGBClassifier(eval_metric='logloss')
}

In [ ]:
# Collect results in a list of dicts so it can be converted to a DataFrame later
model_output = []

for model_name, model in model_dict.items():
    result = scorer(df, model_name, model)
    model_output.append(result)

c:\Users\mabin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [86]:
model_output = pd.DataFrame(model_output)
model_output.sort_values(by='Recall',ascending=False)
# model_output

,Models,Recall,F1-score,ROU-AUC
5,adaboost,0.622609,0.290310,0.724660
4,gradient_boosting,0.502029,0.297339,0.756203
7,xgboost,0.440580,0.292758,0.765086
1,decision_tree,0.412174,0.280418,0.617474
2,random_forest,0.294493,0.265690,0.767392
6,mlp,0.270145,0.346726,0.848103
3,extra_trees,0.219710,0.258791,0.790764
0,svm,0.023188,0.044969,0.839031


To identify the most suitable model for fraud detection, multiple classification algorithms were evaluated using a consistent training and testing framework. Each model was trained within a pipeline that included preprocessing and was tested on a time-based split dataset to avoid data leakage.

The models were evaluated using the following metrics:

Recall (Primary Metric): Since fraud detection aims to identify as many fraudulent transactions as possible, recall was prioritized. A higher recall indicates a lower number of missed fraud cases.

F1-Score: Used to measure the balance between precision and recall.

ROC-AUC Score: Evaluates the model’s ability to distinguish between fraudulent and non-fraudulent transactions across different classification thresholds.

The outcomes :

| Models              | Recall   | F1-score | ROC-AUC |
|---------------------|----------|----------|---------|
| adaboost            | 0.622609 | 0.290310 | 0.724660 |
| gradient_boosting   | 0.502029 | 0.297339 | 0.756203 |
| xgboost             | 0.440580 | 0.292758 | 0.765086 |
| decision_tree       | 0.412174 | 0.280418 | 0.617474 |
| random_forest       | 0.294493 | 0.265690 | 0.767392 |
| mlp                 | 0.270145 | 0.346726 | 0.848103 |
| extra_trees         | 0.219710 | 0.258791 | 0.790764 |
| svm                 | 0.023188 | 0.044969 | 0.839031 |

After comparing all models, AdaBoost achieved the highest recall, while Gradient Boosting demonstrated a strong balance between recall and overall ranking performance (ROC-AUC). Considering both fraud detection requirements and model stability, I am going with Gradient Boosting as the final model.